In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder. \
    appName("pyspark-1"). \
    getOrCreate()

### Read data

In [3]:
df = spark.read.csv("/dataset/nyc-jobs.csv", header=True)
df.printSchema()

root
 |-- Job ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: string (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: string (nullable = true)
 |-- Salary Range To: string (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locatio

In [40]:
df1 = df.filter(col("Job Category").isNotNull())

In [44]:
df1.select("Job Category").distinct().count()

130

In [9]:
#Whats the number of jobs posting per category (Top 10)?

In [11]:
from  pyspark.sql.functions import *

In [46]:
df1.groupBy("Job Category").agg(count("Job ID").alias("Number of Jobs")).orderBy(desc("Number of Jobs")).show(10)

+--------------------+--------------+
|        Job Category|Number of Jobs|
+--------------------+--------------+
|Engineering, Arch...|           504|
|Technology, Data ...|           313|
|       Legal Affairs|           226|
|Public Safety, In...|           182|
|Building Operatio...|           181|
|Finance, Accounti...|           169|
|Administration & ...|           134|
|Constituent Servi...|           129|
|              Health|           125|
|Policy, Research ...|           124|
+--------------------+--------------+
only showing top 10 rows



In [ ]:
# Whats the salary distribution per job category?

In [47]:
from pyspark.sql.functions import col, when, avg, min, max, count

# Calculate estimated salary (average of range)
df_with_avg = df1.withColumn(
    "Estimated Salary",
    (col("Salary Range From") + col("Salary Range To")) / 2
)

# Convert Hourly to Annual (assume 2080 hours/year)
df_converted = df_with_avg.withColumn(
    "Annual Salary",
    when(col("Salary Frequency") == "Hourly", col("Estimated Salary") * 2080)
    .otherwise(col("Estimated Salary"))
)

# Group by Job Category and calculate salary stats
salary_stats = df_converted.groupBy("Job Category").agg(
    count("*").alias("Job Count"),
    avg("Annual Salary").alias("Avg Salary"),
    min("Annual Salary").alias("Min Salary"),
    max("Annual Salary").alias("Max Salary")
).orderBy("Avg Salary", ascending=False)

salary_stats.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------------------+------------------+------------------+
|Job Category                                                                                                                                                                                             |Job Count|Avg Salary        |Min Salary        |Max Salary        |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------------------+------------------+------------------+
|Administration & Human Resources Finance, Accounting, & Procurement Building Operations & Maintenance                                                                                     

In [48]:
# Whats the job posting having the highest salary per agency?

In [57]:
from pyspark.sql.window import Window

In [58]:
window = Window.partitionBy("Agency").orderBy(desc("Annual Salary").alias("max_salary"))

In [62]:
# Apply dense_rank and select columns
ranked_df = df_converted.withColumn("rank", dense_rank().over(window))

In [63]:
ranked_df.show()

Py4JJavaError: An error occurred while calling o587.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 91.0 failed 4 times, most recent failure: Lost task 0.3 in stage 91.0 (TID 2755, 172.18.0.6, executor 0): java.lang.UnsupportedOperationException: Cannot evaluate expression: input[29, double, true] DESC NULLS LAST
	at org.apache.spark.sql.catalyst.expressions.Unevaluable$class.doGenCode(Expression.scala:261)
	at org.apache.spark.sql.catalyst.expressions.SortOrder.doGenCode(SortOrder.scala:62)
	at org.apache.spark.sql.catalyst.expressions.Expression$$anonfun$genCode$2.apply(Expression.scala:108)
	at org.apache.spark.sql.catalyst.expressions.Expression$$anonfun$genCode$2.apply(Expression.scala:105)
	at scala.Option.getOrElse(Option.scala:121)
	at org.apache.spark.sql.catalyst.expressions.Expression.genCode(Expression.scala:105)
	at org.apache.spark.sql.catalyst.expressions.codegen.GenerateOrdering$$anonfun$3.apply(GenerateOrdering.scala:83)

In [55]:
df_converted.groupBy("Agency").agg(max("Annual Salary").alias("max_salary")).orderBy(desc("max_salary")).show()

+--------------------+------------------+
|              Agency|        max_salary|
+--------------------+------------------+
|DEPT OF ENVIRONME...|          218587.0|
|   POLICE DEPARTMENT|          217201.0|
|DISTRICT ATTORNEY...|          191913.0|
|NYC HOUSING AUTHO...|          180000.0|
|NYC EMPLOYEES RET...|          177374.5|
|DEPT OF HEALTH/ME...|         170133.84|
|OFFICE OF THE COM...|          167500.0|
|DEPARTMENT OF INV...|          165000.0|
|      LAW DEPARTMENT|          164104.0|
|DEPARTMENT OF SAN...|          158038.4|
|DEPT OF DESIGN & ...|          151795.0|
|DEPARTMENT FOR TH...|          150371.0|
|TAXI & LIMOUSINE ...|          140000.0|
|DEPARTMENT OF COR...|139105.09600000002|
|DEPT OF INFO TECH...|          138968.0|
|HOUSING PRESERVAT...|          135000.0|
|CONFLICTS OF INTE...|          135000.0|
|FINANCIAL INFO SV...|          135000.0|
|DEPT OF PARKS & R...|          128000.0|
|DEPARTMENT OF PRO...|          127500.0|
+--------------------+------------

### Sample function

In [48]:
def get_salary_frequency(df: DataFrame) -> list:
    row_list = df.select('Salary Frequency').distinct().collect()
    return [row['Salary Frequency'] for row in row_list]

### Example of test function

In [65]:
mock_data = [('A', 'Annual'), ('B', 'Daily')]
expected_result = ['Annual', 'Daily']

In [66]:
def test_get_salary_frequency(mock_data: list, 
                              expected_result: list,
                              schema: list = ['id', 'Salary Frequency']):  
    mock_df = spark.createDataFrame(data = mock_data, schema = schema)
    assert get_salary_frequency(mock_df) == expected_result